# Task 4: Mentor Matching & Intervention Engine

## 🚀 Objective
This notebook demonstrates the **Decision Intelligence** layer of HEPro AI+. We convert student analytics (Scores + Clusters) into practical mentor assignments and automated interventions.

In [1]:
from pathlib import Path
import os

# Robust Path Handling for Notebooks
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load Data
We load the student database (with clusters) and the mentor repository.

In [2]:
import pandas as pd
import numpy as np

students = pd.read_csv(DATA_DIR / 'students_clustered.csv')
mentors = pd.read_csv(DATA_DIR / 'mentors.csv')

print(f"Loaded {len(students)} students and {len(mentors)} mentors.")

Loaded 50 students and 12 mentors.


## 2. Decision Logic: Need Identification
We identify the student's primary 'Pain Point' using a prioritized rule set:
1. **Wellness** (Stress/WWS)
2. **Career** (ML Cluster/CRS)
3. **Academics** (APS)
4. **Productivity** (PTMS)

In [3]:
def get_student_needs(row):
    if row['stress_level'] > 7 or row['WWS'] < 40:
        return 'Wellness', 'Stress Counseling', 'High' if row['SRI'] < 60 else 'Medium'
    if 'Career-Confused' in str(row['cluster_label']) or row['CRS'] < 40:
        return 'Career', 'Career Coaching', 'Medium'
    if row['APS'] < 50:
        return 'Academic', 'Academic Tutoring', 'High'
    if row['PTMS'] < 50:
        return 'Productivity', 'Productivity Coaching', 'Medium'
    if row['SRI'] >= 80:
        return 'Productivity', 'Leadership & Growth', 'Low'
    
    # Fallback based on minimum score
    scores = {'Academic': row['APS'], 'Wellness': row['WWS'], 'Productivity': row['PTMS'], 'Career': row['CRS']}
    min_area = min(scores, key=scores.get)
    return min_area, 'Routine Check-in', 'Low'

## 3. Matching Algorithm (Capacity Aware)
The engine ensures a professional match while preventing mentor burnout.

In [4]:
def match_mentor(need_area, mentors_df, student_category):
    # Match expertise and availability
    candidates = mentors_df[(mentors_df['expertise_area'] == need_area) & 
                            (mentors_df['current_load'] < mentors_df['max_students'])]
    
    if candidates.empty:
        candidates = mentors_df[mentors_df['current_load'] < mentors_df['max_students']]
    if candidates.empty: return "TBD (Waitlist)"
    
    # Prioritize Seniority for High Risk
    if student_category == 'Red':
        seniors = candidates[candidates['experience_level'] == 'Senior']
        if not seniors.empty: candidates = seniors
            
    # Load Balance
    return candidates.sort_values(by='current_load').iloc[0]['mentor_id']

## 4. Final Processing & Alert Simulation

In [5]:
pd.options.mode.chained_assignment = None # Suppress warning
results = []
alert_log = []

for _, student in students.iterrows():
    mentor_type, intervention, priority = get_student_needs(student)
    mentor_id = match_mentor(mentor_type, mentors, student['category'])
    
    if mentor_id != "TBD (Waitlist)":
        mentors.loc[mentors['mentor_id'] == mentor_id, 'current_load'] += 1
    
    # Alert Triggers
    alert_flag = (student['SRI'] < 40 or student['category'] == 'Red' or student['stress_level'] > 8)
    if alert_flag: 
        priority = 'Immediate'
        alert_log.append(f"⚠️ ALERT: Student {student['student_id']} - Reason: High Risk")
        
    results.append({
        'student_id': student['student_id'],
        'cluster': student['cluster_label'],
        'SRI': student['SRI'],
        'category': student['category'],
        'assigned_mentor_id': mentor_id,
        'mentor_type': mentor_type,
        'intervention_type': intervention,
        'priority': priority,
        'alert_flag': alert_flag
    })

final_recommendations = pd.DataFrame(results)
print(f"Process Complete. Created recommendations for {len(final_recommendations)} students.")
final_recommendations.head(10)

Process Complete. Created recommendations for 50 students.


,student_id,cluster,SRI,category,assigned_mentor_id,mentor_type,intervention_type,priority,alert_flag
0,S1001,Career-Confused Underperformers,70.78,Blue,M003,Career,Career Coaching,Medium,False
1,S1002,Career-Confused Underperformers,38.34,Red,M001,Wellness,Stress Counseling,Immediate,True
2,S1003,Career-Confused Underperformers,73.02,Blue,M007,Career,Career Coaching,Medium,False
3,S1004,Career-Confused Underperformers,62.95,Blue,M011,Career,Career Coaching,Medium,False
4,S1005,Stable & Balanced Students,81.49,Green,M004,Productivity,Leadership & Growth,Low,False
5,S1006,Stable & Balanced Students,70.43,Blue,M008,Productivity,Routine Check-in,Low,False
6,S1007,Career-Confused Underperformers,57.07,Yellow,M003,Career,Career Coaching,Medium,False
7,S1008,Stable & Balanced Students,80.47,Green,M012,Productivity,Productivity Coaching,Medium,False
8,S1009,Stable & Balanced Students,65.46,Blue,M004,Productivity,Productivity Coaching,Medium,False
9,S1010,Stable & Balanced Students,64.81,Blue,M008,Productivity,Productivity Coaching,Medium,False


## 5. View Alert Simulation Output

In [6]:
print("--- HIGH-RISK LOG SIMULATION ---")
for alert in alert_log[:5]:
    print(alert)

--- HIGH-RISK LOG SIMULATION ---
⚠️ ALERT: Student S1002 - Reason: High Risk
⚠️ ALERT: Student S1014 - Reason: High Risk
⚠️ ALERT: Student S1024 - Reason: High Risk
⚠️ ALERT: Student S1025 - Reason: High Risk
⚠️ ALERT: Student S1031 - Reason: High Risk
